In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import pandas as pd
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy
import pickle
import os

MODEL_PATH = "./model.pickle"
CSV_PATH = "./financial_dataset.csv"

In [ ]:
from google.colab import files
df = files.upload()

Saving financial_dataset.csv to financial_dataset (1).csv


In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

,Statement,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,$SPY wouldn't be surprised to see a green close,positive
4,Shell's $70 Billion BG Deal Meets Shareholder ...,negative


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,1852
negative,860


In [ ]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()

def get_label(tag):
    if tag.startswith('j'):
        return 'a'
    elif tag.startswith('r') or tag.startswith('v') or tag.startswith('n'):
        return tag[0]
    else:
        return 'a'

def lemmatizing(word_list):
    lemma_list = []
    tagged = pos_tag(word_list)
    for word, tag in tagged:
        label = get_label(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemma_list.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemma_list.append(result)

    return lemma_list

def preprocessing(sentence):
    eng_stopwords = stopwords.words('english')
    punctuations = string.punctuation

    word_list = word_tokenize(sentence)
    removed_stopwords = [token for token in word_list if token not in eng_stopwords]
    removed_punctuation = [token for token in removed_stopwords if token not in punctuations]

    lemma_list = lemmatizing(removed_punctuation)

    return lemma_list

In [ ]:
# Function write statement
def write_statement():
    while True:
        statement = input("Please write your statement: ")
        if(len(statement.split()) < 2):
            print("Invalid input, write at least 2 words")
        else:
            return statement

In [ ]:

sentence = "Hello world, we are not okay today"
preprocessing(sentence)

['Hello', 'world', 'okay', 'today']

In [ ]:
# Analisis Distribusi Frekuensi
x = df["Statement"]
y = df["Sentiment"]

all_sentence = " ".join(x)
all_token = preprocessing(all_sentence)

from nltk import FreqDist
freq_dist = FreqDist(all_token)

# Catatan: Di gambar ada sedikit typo pada print(freq_dist.most_common)
# Jika ingin melihat hasilnya, harusnya menggunakan tanda kurung: print(freq_dist.most_common())
print(freq_dist.most_common)

<bound method Counter.most_common of FreqDist({'EUR': 683, "'s": 473, 'mn': 459, 'profit': 344, 'The': 332, 'sale': 308, 'company': 302, 'say': 289, 'Finnish': 266, 'net': 260, ...})>


In [ ]:
# Fungsi ekstraksi fitur
def extract_feature(statement):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in statement)
    return features

In [ ]:
# Membuat feature sets
feature_sets = []

for (statement, sentiment) in zip(x, y):
    features = extract_feature(preprocessing(statement))
    feature_sets.append((features, sentiment))

print(feature_sets[0])

({'The': True, 'GeoSolutions': True, 'technology': True, 'leverage': True, 'Benefon': True, "'s": True, 'GPS': True, 'solution': True, 'provide': True, 'Location': True, 'Based': True, 'Search': True, 'Technology': True, 'Communities': True, 'Platform': True, 'location': True, 'relevant': True, 'multimedia': True, 'content': True, 'new': True, 'powerful': True, 'commercial': True, 'model': True, 'ESI': False, 'low': False, '1.50': False, '2.50': False, 'BK': False, 'real': False, 'possibility': False, 'For': False, 'last': False, 'quarter': False, '2010': False, 'Componenta': False, 'net': False, 'sale': False, 'double': False, 'EUR131m': False, 'EUR76m': False, 'period': False, 'year': False, 'earlier': False, 'move': False, 'zero': False, 'pre-tax': False, 'profit': False, 'loss': False, 'EUR7m': False, 'SPY': False, 'would': False, "n't": False, 'surprise': False, 'see': False, 'green': False, 'close': False, 'Shell': False, '70': False, 'Billion': False, 'BG': False, 'Deal': False,

In [ ]:
# Membagi data menjadi training dan testing (80:20)
train_count = int(len(feature_sets) * 0.8)
train_set = feature_sets[:train_count]
test_set = feature_sets[train_count:]

In [ ]:
# Function train model
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    test_accuracy = accuracy(classifier, test_set)

    classifier.show_most_informative_features(7)
    print(f"Accuracy: {test_accuracy * 100}%")

    # Simpan model ke file (pastikan MODEL_PATH sudah didefinisikan, misal: MODEL_PATH = 'model.pkl')
    with open(MODEL_PATH, "wb") as f:
        pickle.dump(classifier, f)

    return classifier

In [ ]:
from nltk.corpus import wordnet

def show_pos_tag(statement):
    print(" POS TAGGING: ")
    tokens = preprocessing(statement)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word} : {tag}")
    return tokens

def show_syn_ant(tokens):
    print("syn and ant")
    for token in tokens:
        syno = []
        anto = []
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                syno.append(lemma.name())
                if lemma.antonyms():
                    anto.append(lemma.antonyms()[0].name())
        print(f"Word: {token}")
        print(f"  Synonyms: {set(syno)}")
        print(f"  Antonyms: {set(anto)}")

def analyze_statement(statement, classifier):
    if len(statement) == 0:
        print("no statement")
        return
    tokens = show_pos_tag(statement)
    show_syn_ant(tokens)

    # tambahan
    features = extract_feature(tokens)
    result = classifier.classify(features)
    print(f"\nSentiment Prediction: {result}")

In [ ]:
# --- Bikin Menu ---
statement = ""
classifier = None

while True:
    print("\n1. Write your statement")
    print("2. Analyse your statement")
    print("3. Exit")

    choice = input('>> ')

    if choice == '1':
        statement = write_statement()
        print(f"Current Statement: {statement}")

        # Load model jika ada, jika tidak maka train baru
        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, "rb") as f:
                classifier = pickle.load(f)
        else:
            # Perbaikan: Pakai '=' bukan '==' dan panggil fungsinya ()
            classifier = train_model()

    elif choice == '2':
        if classifier is None:
            print("Please write a statement first to initialize the model!")
        else:
            analyze_statement(statement, classifier)

    elif choice == '3':
        print('Program End')
        break
    else:
        print('Invalid input')


1. Write your statement
2. Analyse your statement
3. Exit
>> 1
Please write your statement: hallo bang
Current Statement: hallo bang
Most Informative Features
                decrease = True           negati : positi =     32.4 : 1.0
                    fell = True           negati : positi =     30.4 : 1.0
                     lay = True           negati : positi =     19.8 : 1.0
                   staff = True           negati : positi =     16.9 : 1.0
                      25 = True           negati : positi =     13.9 : 1.0
                   lower = True           negati : positi =     13.9 : 1.0
                    sign = True           positi : negati =     13.2 : 1.0
Accuracy: 75.87476979742172%

1. Write your statement
2. Analyse your statement
3. Exit
>> 3
Program End


In [ ]:
def show_pos_tag(statement):
    print("POS TAGGING: ")
    tokens = preprocessing(statement)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word} : {tag}")
    return tokens

def show_syn_ant(tokens):
    print("Synonym and Antonym: ")
    for token in tokens:
        synonym = []
        antonym = []

        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonym.append(lemma.name())
                for ant in lemma.antonyms():
                    antonym.append(ant.name())

        if synonym:
            print("Synonym: ")
            print(synonym[:5]) # Ambil 5 saja biar gak kepanjangan
        else:
            print("No synonym")

        if antonym:
            print("Antonym: ")
            print(antonym[:5])
        else:
            print("No antonym")

In [ ]:
def analyze_statement(statement, classifer):
    if len(statement) == 0:
        print("No statement")
        return

    tokens = show_pos_tag(statement)
    show_syn_ant(tokens)

    preprocessed_text = preprocessing(statement)
    extracted = extract_feature(preprocessed_text)
    prediction = classifer.classify(extracted)

    print(f"Predicted Sentiment: {prediction}")

In [ ]:
statement = ""
classifer = None # Perhatikan penulisan 'classifer' sesuai gambar kamu

while True:
    print("1. Write your Statement")
    print("2. Analyze your Statement")
    print("3. Exit")

    choice = input(">> ")

    if (choice == "1"):
        # Function 1
        statement = write_statement()
        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, "rb") as f:
                classifer = pickle.load(f)
        else:
            classifer = train_model()

    elif (choice == "2"):
        # Function 2
        analyze_statement(statement, classifer)

    elif (choice == "3"):
        print("Program Ended...")
        break
    else:
        print("Invalid Input, Please press 1-3")

1. Write your Statement
2. Analyze your Statement
3. Exit
>> 1
Please write your statement: why you gotta hug me like that every time you see me
1. Write your Statement
2. Analyze your Statement
3. Exit
>> 2
POS TAGGING: 
get : VB
ta : JJ
hug : NNS
like : IN
every : DT
time : NN
see : VB
Synonym and Antonym: 
Synonym: 
['get', 'get', 'acquire', 'become', 'go']
Antonym: 
['leave', 'take_away', 'end']
Synonym: 
['tantalum', 'Ta', 'atomic_number_73']
No antonym
Synonym: 
['hug', 'clinch', 'squeeze', 'embrace', 'hug']
No antonym
Synonym: 
['like', 'the_like', 'the_likes_of', 'like', 'ilk']
Antonym: 
['dislike', 'unlike', 'unlike', 'unalike']
Synonym: 
['every', 'every']
No antonym
Synonym: 
['time', 'clip', 'time', 'time', 'time']
No antonym
Synonym: 
['see', 'see', 'understand', 'realize', 'realise']
No antonym
Predicted Sentiment: negative
1. Write your Statement
2. Analyze your Statement
3. Exit
>> 3
Program Ended...
